<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/Day11/House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
#       HOUSE PRICE PREDICTION - AMES HOUSING DATASET
#
#       Models:
#       1. Linear Regression
#       2. Ridge Regression
#       3. Lasso Regression
#
#       Google Colab
#       CSV FILE UPLOAD AT RUNTIME
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    LassoCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 2. UPLOAD CSV FILE AT RUNTIME
# ============================================================

print("=" * 70)
print("UPLOAD YOUR AMES HOUSING CSV FILE")
print("=" * 70)

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nFile loaded successfully!")

print("File name:", file_name)

print("Dataset shape:", df.shape)


# ============================================================
# 3. DISPLAY FIRST 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FIRST 5 ROWS")
print("=" * 70)

print(df.head())


# ============================================================
# 4. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

df.info()


# ============================================================
# 5. CHECK TARGET COLUMN
# ============================================================

TARGET = "SalePrice"

if TARGET not in df.columns:

    print("\nERROR: SalePrice column was not found.")

    print("\nAvailable columns:")

    print(df.columns.tolist())

    raise ValueError(
        "Your CSV must contain a 'SalePrice' column."
    )


# ============================================================
# 6. CHECK MISSING TARGET VALUES
# ============================================================

missing_target = df[TARGET].isnull().sum()

print("\n" + "=" * 70)
print("TARGET CHECK")
print("=" * 70)

print(
    "Missing SalePrice values:",
    missing_target
)

if missing_target > 0:

    print(
        "Rows with missing SalePrice will be removed."
    )

    df = df.dropna(
        subset=[TARGET]
    )


# ============================================================
# 7. TARGET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("TARGET INFORMATION")
print("=" * 70)

print(
    "Target column:",
    TARGET
)

print("\nSalePrice statistics:")

print(
    df[TARGET].describe()
)


# ============================================================
# 8. DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print(
    "Duplicate rows:",
    duplicate_count
)

if duplicate_count > 0:

    df = df.drop_duplicates()

    print(
        "Duplicate rows removed."
    )

else:

    print(
        "No duplicate rows found."
    )


print(
    "Dataset shape after duplicate handling:",
    df.shape
)


# ============================================================
# 9. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


# ============================================================
# 10. REMOVE ID COLUMNS
# ============================================================

id_columns = []

for column in X.columns:

    if column.lower() in [
        "id",
        "pid",
        "pid2"
    ]:

        id_columns.append(column)


if len(id_columns) > 0:

    print("\n" + "=" * 70)
    print("REMOVING ID COLUMNS")
    print("=" * 70)

    print(
        "Removed columns:",
        id_columns
    )

    X = X.drop(
        columns=id_columns
    )

else:

    print(
        "\nNo ID columns found."
    )


# ============================================================
# 11. IDENTIFY NUMERICAL COLUMNS
# ============================================================

numerical_columns = X.select_dtypes(
    include=np.number
).columns.tolist()


# ============================================================
# 12. IDENTIFY CATEGORICAL COLUMNS
# ============================================================

categorical_columns = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


# ============================================================
# 13. DISPLAY FEATURE INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print(
    "Numerical features:",
    len(numerical_columns)
)

print(
    "Categorical features:",
    len(categorical_columns)
)


# ============================================================
# 14. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )

    ]
)


# ============================================================
# 15. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )

    ]
)


# ============================================================
# 16. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )

    ]
)


# ============================================================
# 17. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42
)


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    "Training samples:",
    X_train.shape[0]
)

print(
    "Testing samples:",
    X_test.shape[0]
)


# ============================================================
# 18. SCALE TARGET FOR LASSO
# ============================================================
#
# Lasso is trained using the scaled target.
#
# IMPORTANT:
# We only fit the target scaler using y_train.
# This prevents data leakage.
#
# ============================================================

target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()


# ============================================================
# 19. LINEAR REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING LINEAR REGRESSION")
print("=" * 70)

linear_model = Pipeline(
    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            LinearRegression()
        )

    ]
)


linear_model.fit(
    X_train,
    y_train
)


linear_predictions = linear_model.predict(
    X_test
)


print(
    "Linear Regression training completed."
)


# ============================================================
# 20. RIDGE REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING RIDGE REGRESSION")
print("=" * 70)

ridge_model = Pipeline(
    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            Ridge(
                alpha=1.0
            )
        )

    ]
)


ridge_model.fit(
    X_train,
    y_train
)


ridge_predictions = ridge_model.predict(
    X_test
)


print(
    "Ridge Regression training completed."
)


# ============================================================
# 21. LASSO REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING LASSO REGRESSION")
print("=" * 70)

print(
    "Finding the best alpha using 5-fold cross-validation..."
)


lasso_model = Pipeline(
    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            LassoCV(

                # Search multiple alpha values
                alphas=np.logspace(
                    -4,
                    1,
                    50
                ),

                # 5-fold cross-validation
                cv=5,

                # Maximum iterations
                max_iter=200000,

                # Convergence tolerance
                tol=1e-4,

                # Use all available CPU cores
                n_jobs=-1,

                # Fixed random state
                random_state=42

            )
        )

    ]
)


# ============================================================
# 22. TRAIN LASSO
# ============================================================

lasso_model.fit(
    X_train,
    y_train_scaled
)


# ============================================================
# 23. LASSO PREDICTION
# ============================================================

lasso_predictions_scaled = lasso_model.predict(
    X_test
)


# ============================================================
# 24. CONVERT LASSO PREDICTIONS
#     BACK TO ORIGINAL SALEPRICE SCALE
# ============================================================

lasso_predictions = target_scaler.inverse_transform(

    lasso_predictions_scaled.reshape(
        -1,
        1
    )

).ravel()


print(
    "Lasso Regression training completed."
)


# ============================================================
# 25. BEST LASSO ALPHA
# ============================================================

best_lasso_alpha = (
    lasso_model
    .named_steps["model"]
    .alpha_
)


print("\nBest Lasso alpha:")

print(
    best_lasso_alpha
)


# ============================================================
# 26. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    actual_values,
    predictions
):

    mae = mean_absolute_error(
        actual_values,
        predictions
    )

    mse = mean_squared_error(
        actual_values,
        predictions
    )

    rmse = np.sqrt(
        mse
    )

    r2 = r2_score(
        actual_values,
        predictions
    )

    return {

        "Model": model_name,

        "MAE": mae,

        "MSE": mse,

        "RMSE": rmse,

        "R2 Score": r2

    }


# ============================================================
# 27. EVALUATE LINEAR REGRESSION
# ============================================================

linear_results = evaluate_model(

    "Linear Regression",

    y_test,

    linear_predictions

)


# ============================================================
# 28. EVALUATE RIDGE REGRESSION
# ============================================================

ridge_results = evaluate_model(

    "Ridge Regression",

    y_test,

    ridge_predictions

)


# ============================================================
# 29. EVALUATE LASSO REGRESSION
# ============================================================

lasso_results = evaluate_model(

    "Lasso Regression",

    y_test,

    lasso_predictions

)


# ============================================================
# 30. MODEL COMPARISON
# ============================================================

results = pd.DataFrame(

    [
        linear_results,
        ridge_results,
        lasso_results
    ]

)


print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results.to_string(
        index=False
    )
)


# ============================================================
# 31. BEST MODEL
# ============================================================

best_model_index = results[
    "R2 Score"
].idxmax()


best_model = results.loc[
    best_model_index,
    "Model"
]


print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)

print(
    "Best model based on R2 Score:",
    best_model
)


# ============================================================
# 32. PREDICTION COMPARISON
# ============================================================

prediction_comparison = pd.DataFrame({

    "Actual Price":
        y_test.to_numpy(),

    "Linear Regression":
        linear_predictions,

    "Ridge Regression":
        ridge_predictions,

    "Lasso Regression":
        lasso_predictions

})


print("\n" + "=" * 70)
print("SAMPLE PREDICTIONS")
print("=" * 70)

print(

    prediction_comparison
    .head(10)
    .to_string(index=False)

)


# ============================================================
# 33. SAVE MODEL COMPARISON
# ============================================================

results.to_csv(

    "house_price_model_comparison.csv",

    index=False

)


# ============================================================
# 34. SAVE PREDICTIONS
# ============================================================

prediction_comparison.to_csv(

    "house_price_predictions.csv",

    index=False

)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PROJECT COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "Dataset       : Ames Housing Dataset"
)

print(
    "Target        : SalePrice"
)

print(
    "Problem Type  : Regression"
)

print(
    "Models        : Linear Regression, Ridge, LassoCV"
)

print(
    "Preprocessing  : Missing Values + Encoding + Scaling"
)

print(
    "Lasso          : Target Scaling + Cross Validation"
)

print(
    "Evaluation     : MAE, MSE, RMSE, R2"
)

print(
    "Best Model     :",
    best_model
)

print(
    "Best Lasso Alpha:",
    best_lasso_alpha
)

print("\nFiles created:")

print(
    "1. house_price_model_comparison.csv"
)

print(
    "2. house_price_predictions.csv"
)

print("=" * 70)



UPLOAD YOUR AMES HOUSING CSV FILE


Saving AmesHousing.csv to AmesHousing (1).csv

File loaded successfully!
File name: AmesHousing (1).csv
Dataset shape: (2930, 82)

FIRST 5 ROWS
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0     NaN    NaN          NaN   
1   NaN       Reg          Lvl  ...         0     NaN  MnPrv          NaN   
2   NaN       IR1          Lvl  ...         0     NaN    NaN         Gar2   
3   NaN       Reg          Lvl  ...         0     NaN    NaN          NaN   
4   NaN 